In [6]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

23:25:12 [INFO] Device: cuda


In [7]:
CFG = {
    "task":        "lo",
    "dataset":     "kcnh2",
    "fp_type":     "rdkit_desc",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [ ]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [9]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

23:25:12 [INFO] Loaded lo/kcnh2 fold 1: train=3313, test=406
23:25:12 [INFO] Fold 1 | train=3313 y_mean=5.662 y_std=0.561 | test=406 n_clusters=34
23:25:12 [INFO] Loaded lo/kcnh2 fold 2: train=3313, test=406
23:25:12 [INFO] Fold 2 | train=3313 y_mean=5.663 y_std=0.562 | test=406 n_clusters=34
23:25:12 [INFO] Loaded lo/kcnh2 fold 3: train=3313, test=406
23:25:12 [INFO] Fold 3 | train=3313 y_mean=5.662 y_std=0.561 | test=406 n_clusters=34
23:25:12 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_train_1.npz
23:25:12 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_test_1.npz
23:25:12 [INFO] Fold 1 | X_train: (3313, 217), X_test: (406, 217) | y_mean=5.662 y_std=0.561
23:25:12 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_train_2.npz
23:25:12 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features

In [ ]:
# ── Block 4 ── Nested CV random search ─────────────────────────────────────

logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP KCNH2 Lo — rdkit")

23:25:12 [INFO] Estimated time: ~30 min
23:25:12 [INFO] 
OUTER FOLD 1
23:25:15 [INFO]   [1/100] inner Spearman=0.0430 (3s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
23:25:18 [INFO]   [2/100] inner Spearman=0.0257 (3s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
23:25:20 [INFO]   [3/100] inner Spearman=0.0338 (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
23:25:21 [INFO]   [4/100] inner Spearman=0.0145 (1s) | L=2 N=128 r=0.7 dropout=0.3 lr=3e-03
23:25:22 [INFO]   [5/100] inner Spearman=0.0145 (1s) | L=2 N=256 r=1.0 dropout=0.6 lr=3e-03
23:25:23 [INFO]   [6/100] inner Spearman=-0.0145 (1s) | L=1 N=128 r=0.7 dropout=0.6 lr=1e-04
23:25:24 [INFO]   [7/100] inner Spearman=0.0251 (2s) | L=3 N=256 r=1.0 dropout=0.5 lr=1e-03
23:25:25 [INFO]   [8/100] inner Spearman=0.0145 (1s) | L=2 N=128 r=0.7 dropout=0.5 lr=5e-04
23:25:26 [INFO]   [9/100] inner Spearman=0.0145 (1s) | L=1 N=256 r=0.5 dropout=0.5 lr=1e-04
23:25:30 [INFO]   [10/100] inner Spearman=0.0232 (3s) | L=3 N=128 r=0.5 dropout=0.0 lr=5e-04
23:25:3

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 42])